
# Fabric Provisioning Notebook

This notebook provisions Fabric artifacts and connections in a specified workspace.

✅ **Part A** provisions *artifacts* and SQL connections using **sempy_labs**.  
✅ **Part B** provisions *lakehouses and other connections* using **REST calls**.

Overwrite behavior is controlled with `overwrite=False`, printing friendly messages for existing items. \
**User needs Fabric Admin role to run this notebook**.
    

In [ ]:
# Optional: Install if needed
# !pip install --upgrade semantic-link-labs

In [ ]:
import sempy_labs
import requests
import notebookutils
from notebookutils import mssparkutils, credentials

## 📌 Parameters

In [ ]:
# Workspace and Artifacts
workspace_name = "Admin Monitoring"
bronze_lakehouse_name = "Bronze_Lake"
silver_lakehouse_name = "Silver_Lake"
warehouse_name = "Gold_EDW"
sql_db_name = "UTIL_Database"

# ADLS Gen2
storage_url = "https://<your-storage-account>.dfs.core.windows.net/<your-container>"
adls_connection_name = "ADLSConn"

# Azure SQL
azure_sql_conn_name = "AzureSQLConn"
azure_sql_server = "<your-server>.database.windows.net"
azure_sql_db = "<your-db-name>"
azure_sql_user = "<username>"
azure_sql_pass = "<password>"

# SQL Server (on-prem via Gateway)
sql_server_conn_name = "OnPremSQLConn"
sql_server_name = "<sql-server-host>"
sql_server_db = "<sql-db-name>"
sql_server_user = "<username>"
sql_server_pass = "<password>"
gateway_id = "<your-gateway-id>"

# ODBC
odbc_conn_name = "OdbcConn"
odbc_conn_string = "DSN=MyOdbcSource;UID=user;PWD=pass"

# Dynamics 365 (CRM)
dynamics_crm_conn_name = "DynamicsCrmConn"
dynamics_crm_url = "https://<org>.crm.dynamics.com"
dynamics_crm_client_id = "<client-id>"
dynamics_crm_client_secret = "<client-secret>"
dynamics_crm_tenant_id = "<tenant-id>"

# Business Central
bc_conn_name = "BusinessCentralConn"
bc_url = "https://api.businesscentral.dynamics.com/v2.0/<tenant-id>/sandbox/ODataV4/"
bc_client_id = "<client-id>"
bc_client_secret = "<client-secret>"
bc_tenant_id = "<tenant-id>"

## 📌 Resolve Workspace ID

In [ ]:
workspace_df = sempy_labs.admin.list_workspaces(workspace=workspace_name)
if workspace_df.empty:
    raise ValueError(f"No workspace found with name '{workspace_name}'")
workspace_id = workspace_df.iloc[0]['Id']
print(f"✅ Resolved Workspace ID: {workspace_id}")


## ✅ Part A: Provision Artifacts and SQL Connections with sempy_labs

Explicit calls for:
- Warehouse
- SQL Database
- Azure SQL Connection (Cloud)
- On-Prem SQL Server Connection
    

In [ ]:
# Create Warehouse
warehouse_result = sempy_labs.create_warehouse(
    warehouse_name,
    workspace=workspace_id
)
print(f"✅ Warehouse provisioned: {warehouse_name}")

In [ ]:
# Create SQL Database
sql_database_result = sempy_labs.create_sql_database(
    sql_db_name,
    workspace=workspace_id
)
print(f"✅ SQL Database provisioned: {sql_db_name}")

In [ ]:
# Create Azure SQL Connection (Cloud)
azure_sql_conn = sempy_labs.create_cloud_connection(
    workspace_id,
    azure_sql_conn_name,
    "AzureSQLDatabase",
    {
        "server_name": azure_sql_server,
        "database_name": azure_sql_db,
        "user_name": azure_sql_user,
        "password": azure_sql_pass
    }
)
print(f"✅ Azure SQL Connection provisioned: {azure_sql_conn_name}")

In [ ]:
# Create On-Prem SQL Server Connection
sql_server_conn = sempy_labs.create_on_prem_connection(
    workspace_id,
    sql_server_conn_name,
    "SQLServer",
    gateway_id,
    {
        "server_name": sql_server_name,
        "database_name": sql_server_db,
        "user_name": sql_server_user,
        "password": sql_server_pass
    }
)
print(f"✅ On-Prem SQL Server Connection provisioned: {sql_server_conn_name}")


## ✅ Part B: Provision Lakehouses and Other Connections with REST Calls

- Gets token with analysis.windows.net/powerbi/api scope
- Builds a single requests.Session with Authorization header
- Uses artifact_exists helper
- Bronze/Silver Lakehouse creation exactly as you provided
- Same session for other connections
    

### 📌 Part B.1: REST Setup

In [ ]:
# Get Access Token and Setup Session
fmd_api_access_token = notebookutils.credentials.getToken('https://analysis.windows.net/powerbi/api')
if not fmd_api_access_token:
    raise Exception("Failed to get access token")

def create_fabric_session(fabric_token):
    fabric_headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {fabric_token}'
    }
    session = requests.Session()
    session.headers.update(fabric_headers)
    return session

session = create_fabric_session(fmd_api_access_token)

### 📌 Part B.2: artifact_exists Helper

In [ ]:
def artifact_exists(url, name):
    response = session.get(url)
    if response.status_code == 200:
        items = response.json().get('value', [])
        return any(item.get('name') == name for item in items)
    return False

### 📌 Part B.3: Lakehouse Provisioning

In [ ]:
# ✅ Create Bronze Lakehouse
lakehouse_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses"
if not artifact_exists(lakehouse_url, bronze_lakehouse_name):
    payload = {"name": bronze_lakehouse_name, "type": "Lakehouse", "displayName": bronze_lakehouse_name}
    resp = session.post(lakehouse_url, json=payload)
    print("Lakehouse created:", resp.status_code, resp.text)
else:
    print("Lakehouse already exists.")

In [ ]:
# ✅ Create Silver Lakehouse
lakehouse_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses"
if not artifact_exists(lakehouse_url, silver_lakehouse_name):
    payload = {"name": silver_lakehouse_name, "type": "Lakehouse", "displayName": silver_lakehouse_name}
    resp = session.post(lakehouse_url, json=payload)
    print("Lakehouse created:", resp.status_code, resp.text)
else:
    print("Lakehouse already exists.")

### 📌 Part B.4: Other Connections with Same Session

In [ ]:
# Example: ADLS Gen2 Connection
connections_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/connections"
payload = {
    "name": adls_connection_name,
    "type": "ADLSGen2",
    "connectionParameters": {
        "url": storage_url
    }
}
resp = session.post(connections_url, json=payload)
print("ADLS Connection:", resp.status_code, resp.text)